# Emissions

In [1]:
import geopandas as gpd
import pandas as pd
from pandas import Series


def read_links(input_path) -> gpd.GeoDataFrame:
    df_links = pd.read_csv(
        input_path,
        sep=";",
        compression="zstd",
        dtype={"link": "string"},
        low_memory=False,
    )

    links = gpd.GeoDataFrame(
        df_links[~df_links["link"].str.contains("pt_")],
        geometry=gpd.GeoSeries.from_wkt(df_links["geometry"], crs="EPSG:25832"),
    )

    return links


def read_emissions(input_path) -> pd.DataFrame:
    return pd.read_csv(input_path, sep=",", low_memory=False, dtype={"link": "string"})


def filter_links_by_zone(links: gpd.GeoDataFrame, zone: gpd.GeoDataFrame, invert=False) -> Series:
    links_crs = "EPSG:25832"

    zone_to_links_crs = zone.to_crs(links_crs)

    # filter all rows in links whose geometry is fully in zone and return the filtered links
    if invert:
        filtered_links = links[~links.geometry.within(zone_to_links_crs.union_all())]
    else:
        filtered_links = links[links.geometry.within(zone_to_links_crs.union_all())]
    return filtered_links

In [2]:
path_base_links = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-baseCaseCtdExtended/berlin-v6.4.output_links.csv.zst"
path_base_emissions = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-baseCaseCtdExtended/analysis/emissions/emissions_per_link.csv"

path_policy_links = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-policy/berlin-v6.4.output_links.csv.zst"
path_policy_emissions = "/Users/paulh/runs-svn/matsim-berlin/autofrei/1pct-v6.4/berlin-autofrei-v6.4-policy/analysis/emissions/emissions_per_link.csv"

links = read_links(path_policy_links)
zone = gpd.read_file('../../../../input/v6.4/umweltzone/Umweltzone_Berlin.shp')
bezirke = gpd.read_file("/Users/paulh/git/matsim-berlin/input/v6.4/bezirksgrenzen.geojson")

berlin_links = filter_links_by_zone(links, bezirke)
inner_city_links = filter_links_by_zone(berlin_links, zone)

In [6]:
def sum_emissions(emissions_path) -> list[Series]:
    emissions = read_emissions(emissions_path)
    # filter emissions by inner city links
    emissions_inner_city = emissions[emissions["linkId"].isin(inner_city_links["link"])]
    # outer city: linkId in berlin_links but not in inner_city_links
    emissions_outer_city = emissions[
        emissions["linkId"].isin(berlin_links["link"]) & ~emissions["linkId"].isin(inner_city_links["link"])]
    # sum each column in emissions_inner_city and print the result
    return [emissions_inner_city.sum(), emissions_outer_city.sum()]


base_emissions_inner, base_emissions_outer = sum_emissions(path_base_emissions)
policy_emissions_inner, policy_emissions_outer = sum_emissions(path_policy_emissions)

# create table with columns "type", "inner base", "inner policy", "outer base", "outer policy"
emissions_comparison = pd.DataFrame({
    "type": base_emissions_inner.index,
    "inner base": base_emissions_inner.values,
    "inner policy": policy_emissions_inner.values,
    "outer base": base_emissions_outer.values,
    "outer policy": policy_emissions_outer.values,
})

# print emissions table as latex: filter for type CO2_TOTAL and print with cols "inner", "outer" and rows "base", "policy"
co2_row = emissions_comparison.loc[emissions_comparison["type"] == "CO2_TOTAL"].iloc[0]

emissions_comparison_filtered = pd.DataFrame(
    {
        "Base [t]": [
            co2_row["inner base"],
            co2_row["outer base"],
            co2_row["inner base"] + co2_row["outer base"],
        ],
        "Policy [t]": [
            co2_row["inner policy"],
            co2_row["outer policy"],
            co2_row["inner policy"] + co2_row["outer policy"],
        ],
    },
    index=["IR", "OR", "Berlin"],
)

emissions_comparison_filtered[["Base [t]", "Policy [t]"]] = emissions_comparison_filtered[
                                                                ["Base [t]", "Policy [t]"]] / 1_000_000
emissions_comparison_filtered["Rel. change [\\%]"] = ((emissions_comparison_filtered["Policy [t]"] /
                                                       emissions_comparison_filtered["Base [t]"]) - 1.0
                                                      ) * 100.0

latex_table = emissions_comparison_filtered.to_latex(
    float_format="%.1f",
    column_format="lccc",
    caption="CO$_2$ emissions per day on respective links.",
    label="tab:emissions_comparison"
)

latex_table = latex_table.replace(r"\begin{table}", r"\begin{table}\centering")

print(latex_table)


\begin{table}\centering
\caption{CO$_2$ emissions per day on respective links.}
\label{tab:emissions_comparison}
\begin{tabular}{lccc}
\toprule
 & Base [t] & Policy [t] & Rel. change [\%] \\
\midrule
IR & 1096.2 & 868.8 & -20.7 \\
OR & 4907.0 & 4855.1 & -1.1 \\
Berlin & 6003.2 & 5723.9 & -4.7 \\
\bottomrule
\end{tabular}
\end{table}



NameError: name 'emissions' is not defined